# 03 — Propose template change

Given the diagnosed failure patterns and the current template, propose a concrete change with rationale and expected effect on each G-Eval criterion.

**Input:** `02-diagnosis-summary.md`, current `paper_brief_template.md`.

**Output:** `03-proposed-change.md` — the proposal. Does **not** modify the template file.

See [paper-brief-improvement.md](../../docs/specs/paper-brief-improvement.md) step 3.

In [1]:
# Chat model id for the proposal call.
# Use the API model name, not the run folder slug. Example: "gemma4:e4b" (not "gemma4-e4b").
# Leave empty to infer from RUN_ID.
MODEL = ""

# Improvement run folder under data/paper_brief_improvement/.
# Leave empty to use the latest folder that has 02-diagnosis-summary.md.
RUN_ID = ""

In [2]:
from __future__ import annotations

import os
import re
from pathlib import Path

from IPython.display import Markdown, display


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
IMPROVEMENT_PARENT = REPO_ROOT / "data" / "paper_brief_improvement"

print(f"repo root: {REPO_ROOT}")
print(f"improvement parent: {IMPROVEMENT_PARENT}")

repo root: /workspace
improvement parent: /workspace/data/paper_brief_improvement


In [3]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")
CRITERIA = ("faithfulness", "completeness", "conciseness", "topic_agnostic")


def resolve_run_dir(run_id: str) -> Path:
    if run_id:
        d = IMPROVEMENT_PARENT / run_id
        if not (d / "02-diagnosis-summary.md").is_file():
            raise FileNotFoundError(f"No 02-diagnosis-summary.md in {d}")
        return d
    candidates = sorted(
        (
            p.parent
            for p in IMPROVEMENT_PARENT.glob("*/02-diagnosis-summary.md")
            if _RUN_ID_PATTERN.match(p.parent.name)
        ),
        key=lambda d: d.name,
    )
    if not candidates:
        raise FileNotFoundError(
            "No improvement runs with 02-diagnosis-summary.md found under "
            + str(IMPROVEMENT_PARENT)
        )
    return candidates[-1]


def model_slug_from_run_id(run_id: str) -> str | None:
    match = _RUN_ID_PATTERN.match(run_id)
    if not match:
        return None
    _, _, slug = run_id.partition("_")
    return slug or None


def chat_model_from_run_slug(slug: str) -> str:
    if ":" in slug:
        return slug
    if "-" not in slug:
        return slug
    return slug.replace("-", ":", 1)


def resolve_chat_model(model: str, run_id: str) -> str:
    slug = model_slug_from_run_id(run_id)
    candidate = model.strip()
    if not candidate and slug:
        inferred = chat_model_from_run_slug(slug)
        print(f"MODEL empty; inferred {inferred!r} from run_id slug {slug!r}")
        return inferred
    if not candidate:
        raise ValueError(
            "MODEL is required. Set the chat model id (example: 'gemma4:e4b'). "
            "Do not use the run folder slug ('gemma4-e4b')."
        )
    if slug and candidate == slug and ":" not in candidate:
        inferred = chat_model_from_run_slug(slug)
        print(
            f"MODEL looks like a run_id slug ({candidate!r}); "
            f"using chat model id {inferred!r} instead"
        )
        return inferred
    return candidate

In [4]:
run_dir = resolve_run_dir(RUN_ID)
run_id = run_dir.name
model = resolve_chat_model(MODEL, run_id)
os.environ["OPENAI_MODEL"] = model
print(f"chat model: {model}")
print(f"run dir: {run_dir.relative_to(REPO_ROOT)}")

diagnosis_summary_path = run_dir / "02-diagnosis-summary.md"
diagnosis_summary = diagnosis_summary_path.read_text(encoding="utf-8")
print(f"diagnosis summary: {len(diagnosis_summary)} chars")

from paper_reviewer.topic_scope.generate_paper_brief.llm import (
    load_paper_brief_template,
)

template_text = load_paper_brief_template()
print(f"template length: {len(template_text)} chars")

MODEL empty; inferred 'gemma4:e4b' from run_id slug 'gemma4-e4b'
chat model: gemma4:e4b
run dir: data/paper_brief_improvement/20260818T221210Z_gemma4-e4b
diagnosis summary: 2109 chars
template length: 3678 chars


In [5]:
from paper_reviewer.topic_scope.generate_paper_brief.llm import (
    resolve_openai_base_url,
    resolve_openai_model,
)


def _make_client():
    from openai import OpenAI

    api_key = os.environ.get("OPENAI_API_KEY") or "placeholder"
    base_url = resolve_openai_base_url(
        os.environ.get("OPENAI_BASE_URL"),
        in_container=Path("/.dockerenv").exists(),
    )
    return OpenAI(api_key=api_key, base_url=base_url)


def _chat(client, system: str, user: str) -> str:
    resolved_model = resolve_openai_model(os.environ.get("OPENAI_MODEL"))
    resp = client.chat.completions.create(
        model=resolved_model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content or ""


client = _make_client()
print("LLM client ready")

LLM client ready


In [6]:
PROPOSAL_SYSTEM = f"""You are a prompt-engineering specialist for scientific paper briefs.

You receive:
1. A diagnosis summary of common failure patterns in generated paper briefs.
2. The current brief template (system prompt) used to generate those briefs.

Your task: propose ONE concrete change to the template that addresses the diagnosed weaknesses.
Keep the change minimal and targeted. Do not rewrite the entire template unless necessary.

Output Markdown with these sections (use exactly these headings):

## Rationale
Why this change addresses the diagnosed failure patterns.

## Proposed change
Show the change as old text / new text blocks, or as a rewritten template section.
Be precise enough that a developer can apply the change manually.

## Expected effect on G-Eval criteria
For each of the four criteria ({', '.join(CRITERIA)}), state whether the change
should improve, have no effect, or risk degrading the score, and briefly why.

Do NOT apply the change. Do NOT output the full template unless a section rewrite is the proposal."""


proposal_user = (
    "## Diagnosis summary\n\n"
    f"{diagnosis_summary}\n\n"
    "## Current brief template\n\n"
    f"{template_text}"
)

print("proposal prompt ready")

proposal prompt ready


In [7]:
print("generating proposal ...", flush=True)
proposal_text = _chat(client, PROPOSAL_SYSTEM, proposal_user)
print(f"proposal: {len(proposal_text)} chars")

generating proposal ...
proposal: 2264 chars


In [8]:
proposal_path = run_dir / "03-proposed-change.md"
proposal_path.write_text(proposal_text, encoding="utf-8")
print(f"wrote {proposal_path.relative_to(REPO_ROOT)}")

display(Markdown(proposal_text))

wrote data/paper_brief_improvement/20260818T221210Z_gemma4-e4b/03-proposed-change.md


## Rationale
The current instructions for `key_findings` are too passive; they merely ask the model to list "primary metrics or results." However, high-quality scientific briefs (as demonstrated by the diagnosis summary) do not just list data points; they synthesize a finding *and* its immediate implication. By modifying this section's prompt, we force the model to act as an interpreter of the raw data, ensuring that every listed key finding is presented with enough context or significance statement derived from the text (e.g., "X was found in Y areas, suggesting Z risk"). This elevates the brief from a mere summary of results to a true synthesis of scientific impact.

## Proposed change
Modify the description for `key_findings` within the `# Output fields` section.

**Old Text:**
```markdown
### `key_findings` (required)

A short list (typically two or three items) of primary metrics or results. Do not dump every table.
```

**New Text:**
```markdown
### `key_findings` (required)

A short, synthesized list (typically two to four items) detailing the most impactful primary metrics or results. Each finding must state the result and its immediate significance or implication as described by the authors in the text. Do not simply list data points; synthesize the key takeaway.
```

## Expected effect on G-Eval criteria
**faithfulness:** Improve. By requiring the model to explicitly link a finding to its *significance* (which must still be grounded in the source text), we force a deeper, more accurate understanding of the paper's core message, reducing the risk of listing isolated data points without context.

**completeness:** Improve. The change guides the model toward capturing the full scope of an impactful finding—the result *and* its meaning—making the brief feel richer and more complete in conveying scientific insight.

**conciseness:** No effect. While the instruction is slightly longer, it improves information density by ensuring that every bullet point carries both factual weight and interpretive value, maintaining conciseness without sacrificing depth.

**topic_agnostic:** No effect. The change relates purely to the structural requirement of synthesis for a field, not the nature or topic of the science being summarized.